# Large-Scale Protein Feature Enrichment

Enriches **Davis**, **KIBA**, and **BindingDB-KD** DTI parquet datasets with UniProt metadata and sequence descriptors (AAC / PAAC / CTD).

- Raw inputs: `data/raw/{dataset}.parquet`
- Test run (10 rows): `data/testrun/`
- Full run: `data/processed/`
- Davis `Target_ID` gene symbols are mapped to UniProt accessions before fetch.


In [ ]:
%pip install -q requests pandas numpy tqdm propy3 biopython pyarrow


In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)


## Core pipeline

UniProt fetch/cache, metadata summarization, AAC/PAAC/CTD descriptors, Davis gene-symbol mapping, and dataset enrichment.


In [ ]:
from __future__ import annotations



import json

import shutil

import time

from pathlib import Path

from typing import Any, Dict, List, Optional, Tuple



import numpy as np

import pandas as pd

import requests

from Bio.SeqUtils.ProtParam import ProteinAnalysis

from tqdm import tqdm



# --- Paths (overridden by notebook init) ---

ROOT = Path.cwd()

DATA_DIR = ROOT / "data"

RAW_DIR = DATA_DIR / "raw"

LEGACY_RAW_DIR = ROOT / "raw"

CACHE_DIR = DATA_DIR / "cache"

UNIPROT_JSON_CACHE = CACHE_DIR / "uniprot_json"

DAVIS_MAP_CACHE = CACHE_DIR / "davis_target_to_uniprot.tsv"



UNIPROT_REST_BASE = "https://rest.uniprot.org/uniprotkb"

UNIPROT_SEARCH_BASE = "https://rest.uniprot.org/uniprotkb/search"



DATASETS = ["davis", "kiba", "bindingdb_kd"]



REQUEST_DELAY_S = 0.35

FORCE_REFETCH = False





def ensure_data_dirs() -> None:

    for d in [RAW_DIR, CACHE_DIR, UNIPROT_JSON_CACHE, DATA_DIR / "testrun", DATA_DIR / "processed"]:

        d.mkdir(parents=True, exist_ok=True)

    for name in DATASETS:

        dst = RAW_DIR / f"{name}.parquet"

        if dst.exists():

            continue

        src = LEGACY_RAW_DIR / f"{name}.parquet"

        if src.exists():

            shutil.copy2(src, dst)





# --- UniProt fetch ---





def fetch_uniprot_json(

    uniprot_id: str, *, timeout_s: int = 30, max_retries: int = 3

) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:

    url = f"{UNIPROT_REST_BASE}/{uniprot_id}.json"

    last_err: Optional[str] = None

    for attempt in range(1, max_retries + 1):

        try:

            resp = requests.get(

                url, timeout=timeout_s, headers={"User-Agent": "ProteinEnrichment/1.0"}

            )

            if resp.status_code == 200:

                try:

                    return resp.json(), None

                except Exception as e:

                    return None, f"json_decode_error: {e}"

            if resp.status_code in (404, 400):

                return None, f"http_{resp.status_code}"

            last_err = f"http_{resp.status_code}"

        except requests.RequestException as e:

            last_err = f"network_error: {e}"

        if attempt < max_retries:

            time.sleep(0.6 * (2 ** (attempt - 1)))

    return None, last_err or "unknown_error"





def _cache_path_for_id(uniprot_id: str) -> Path:

    safe = uniprot_id.replace("/", "_").replace("\\", "_")

    return UNIPROT_JSON_CACHE / f"{safe}.json"





def get_uniprot_record_cached(

    uniprot_id: str, *, force_refetch: bool = False

) -> Tuple[Optional[Dict[str, Any]], Optional[str], bool]:

    cache_path = _cache_path_for_id(uniprot_id)

    if not force_refetch and cache_path.exists():

        try:

            return json.loads(cache_path.read_text(encoding="utf-8")), None, True

        except Exception as e:

            record, err = fetch_uniprot_json(uniprot_id)

            if record is not None:

                cache_path.write_text(json.dumps(record, ensure_ascii=False), encoding="utf-8")

                return record, None, False

            return None, f"cache_read_error_then_{err or e}", False



    record, err = fetch_uniprot_json(uniprot_id)

    if record is not None:

        cache_path.write_text(json.dumps(record, ensure_ascii=False), encoding="utf-8")

        return record, None, False

    return None, err, False





# --- Summarize ---





def _get(d: Any, path: List[Any], default=None):

    cur = d

    for p in path:

        try:

            if isinstance(p, int):

                cur = cur[p]

            else:

                cur = cur.get(p)

        except Exception:

            return default

        if cur is None:

            return default

    return cur





def _extract_crossref_ids(record: Dict[str, Any], db: str) -> List[str]:

    out: List[str] = []

    for x in record.get("uniProtKBCrossReferences", []) or []:

        if x.get("database") == db:

            pid = x.get("id")

            if pid:

                out.append(pid)

    return out





def _extract_comment_texts(record: Dict[str, Any], comment_type: str) -> List[str]:

    texts: List[str] = []

    for c in record.get("comments", []) or []:

        if c.get("commentType") != comment_type:

            continue

        if "texts" in c:

            for t in c.get("texts", []) or []:

                v = t.get("value") if isinstance(t, dict) else None

                if v:

                    texts.append(v)

        if "text" in c and isinstance(c.get("text"), dict):

            v = c["text"].get("value")

            if v:

                texts.append(v)

        if "subcellularLocations" in c:

            for sl in c.get("subcellularLocations", []) or []:

                loc = _get(sl, ["location", "value"]) or _get(sl, ["location", "id"])

                if loc:

                    texts.append(str(loc))

    return texts





def summarize_uniprot_record(uniprot_id: str, record: Dict[str, Any]) -> Dict[str, Any]:

    seq = _get(record, ["sequence", "value"], "") or ""

    seq_len = _get(record, ["sequence", "length"], None)

    checksum = _get(record, ["sequence", "checksum"], None)



    mw = None

    if seq:

        try:

            mw = float(ProteinAnalysis(seq).molecular_weight())

        except Exception:

            mw = None



    rec_name = _get(record, ["proteinDescription", "recommendedName", "fullName", "value"], None)

    ec_numbers: List[str] = []

    for ec in _get(record, ["proteinDescription", "recommendedName", "ecNumbers"], []) or []:

        v = ec.get("value") if isinstance(ec, dict) else None

        if v:

            ec_numbers.append(v)



    go_terms: List[str] = []

    for go in record.get("uniProtKBCrossReferences", []) or []:

        if go.get("database") == "GO":

            gid = go.get("id")

            if gid:

                go_terms.append(gid)



    keywords: List[str] = []

    for kw in record.get("keywords", []) or []:

        kid = kw.get("id") or kw.get("value")

        if kid:

            keywords.append(str(kid))



    reactome_ids = _extract_crossref_ids(record, "Reactome")

    ft = record.get("features", []) or []

    ft_types = [f.get("type") for f in ft if isinstance(f, dict)]



    def _count(t: str) -> int:

        return int(sum(1 for x in ft_types if x == t))



    subcell_texts = _extract_comment_texts(record, "SUBCELLULAR LOCATION")

    function_texts = _extract_comment_texts(record, "FUNCTION")

    pathway_texts = _extract_comment_texts(record, "PATHWAY")

    enzyme_reg_texts = _extract_comment_texts(record, "ENZYME REGULATION")

    tissue_texts = _extract_comment_texts(record, "TISSUE SPECIFICITY")

    dev_texts = _extract_comment_texts(record, "DEVELOPMENTAL STAGE")

    pdb_ids = _extract_crossref_ids(record, "PDB")



    isoform_count = 0

    for c in _get(record, ["comments"], []) or []:

        if isinstance(c, dict) and c.get("commentType") == "ALTERNATIVE PRODUCTS":

            isoform_count += len(c.get("isoforms", []) or [])



    return {

        "uniprot_id": uniprot_id,

        "sequence": seq,

        "sequence_length": seq_len if seq_len is not None else (len(seq) if seq else None),

        "sequence_checksum": checksum,

        "molecular_weight": mw,

        "recommended_name": rec_name,

        "ec_numbers": ";".join(sorted(set(ec_numbers))) if ec_numbers else None,

        "go_terms": ";".join(sorted(set(go_terms))) if go_terms else None,

        "keywords": ";".join(sorted(set(keywords))) if keywords else None,

        "reactome": ";".join(sorted(set(reactome_ids))) if reactome_ids else None,

        "subcellular_location": ";".join(sorted(set(subcell_texts))) if subcell_texts else None,

        "function": " ".join(function_texts) if function_texts else None,

        "pathway": " ".join(pathway_texts) if pathway_texts else None,

        "enzyme_regulation": " ".join(enzyme_reg_texts) if enzyme_reg_texts else None,

        "tissue_specificity": " ".join(tissue_texts) if tissue_texts else None,

        "developmental_stage": " ".join(dev_texts) if dev_texts else None,

        "isoform_count": isoform_count,

        "ft_transmem_count": _count("TRANSMEM"),

        "ft_topo_dom_count": _count("TOPO_DOM"),

        "ft_domain_count": _count("DOMAIN"),

        "ft_region_count": _count("REGION"),

        "ft_binding_count": _count("BINDING"),

        "ft_ptm_count": _count("MOD_RES") + _count("CARBOHYD"),

        "ft_variant_count": _count("VARIANT"),

        "ft_mutagen_count": _count("MUTAGEN"),

        "pdb_count": len(pdb_ids),

        "pdb_ids": ";".join(sorted(set(pdb_ids))) if pdb_ids else None,

    }





# --- Descriptors ---



AA20 = list("ACDEFGHIKLMNPQRSTVWY")



try:

    from propy import AAComposition

    from propy import CTD as PropyCTD

    from propy import PseudoAAC



    _HAVE_PROPY = True

except Exception:

    _HAVE_PROPY = False





def _aac_fallback(seq: str) -> Dict[str, float]:

    seq = seq.upper()

    n = len(seq)

    if n == 0:

        return {f"aac_{aa}": np.nan for aa in AA20}

    counts = {aa: 0 for aa in AA20}

    for ch in seq:

        if ch in counts:

            counts[ch] += 1

    return {f"aac_{aa}": counts[aa] / n for aa in AA20}





def _paac_fallback(seq: str, lambda_: int = 10, weight: float = 0.05) -> Dict[str, float]:

    aac = _aac_fallback(seq)

    seq = seq.upper()

    dipep_keys = ["AA", "AC", "CA", "CC", "GG", "PP", "RR", "SS", "TT", "VV"]

    dipep = {f"paac_dipep_{k}": 0.0 for k in dipep_keys}

    if len(seq) >= 2:

        total = len(seq) - 1

        for i in range(total):

            k = seq[i : i + 2]

            if k in dipep_keys:

                dipep[f"paac_dipep_{k}"] += 1.0

        for k in dipep_keys:

            dipep[f"paac_dipep_{k}"] /= total

    paac = {f"paac_{k.replace('aac_', '')}": float(v) for k, v in aac.items()}

    paac.update(dipep)

    paac["paac_lambda"] = float(lambda_)

    paac["paac_weight"] = float(weight)

    return paac





def _ctd_fallback(seq: str) -> Dict[str, float]:

    seq = "".join([c for c in seq.upper() if c.isalpha()])

    if not seq:

        return {"ctd_hydro_C1": np.nan, "ctd_hydro_C2": np.nan, "ctd_hydro_C3": np.nan}

    g1, g2, g3 = set("RKEDQN"), set("GASTPHY"), set("CLVIMFW")

    groups = []

    for aa in seq:

        if aa in g1:

            groups.append(1)

        elif aa in g2:

            groups.append(2)

        elif aa in g3:

            groups.append(3)

    if not groups:

        return {"ctd_hydro_C1": np.nan, "ctd_hydro_C2": np.nan, "ctd_hydro_C3": np.nan}

    n = len(groups)

    return {

        "ctd_hydro_C1": float(sum(1 for x in groups if x == 1) / n),

        "ctd_hydro_C2": float(sum(1 for x in groups if x == 2) / n),

        "ctd_hydro_C3": float(sum(1 for x in groups if x == 3) / n),

    }





def compute_descriptors(uniprot_id: str, sequence: str) -> Dict[str, float]:

    if not sequence:

        return {"uniprot_id": uniprot_id}

    features: Dict[str, float] = {"uniprot_id": uniprot_id}

    if _HAVE_PROPY:

        try:

            aac = AAComposition.CalculateAAComposition(sequence)

            for k, v in aac.items():

                features[f"aac_{k}"] = float(v)

        except Exception:

            features.update(_aac_fallback(sequence))

        try:

            paac = PseudoAAC.GetAPseudoAAC(sequence, lamda=10, weight=0.05)

            for k, v in paac.items():

                features[f"paac_{k}"] = float(v)

        except Exception:

            features.update(_paac_fallback(sequence))

        try:

            ctd = PropyCTD.CalculateCTD(sequence)

            for k, v in ctd.items():

                features[f"ctd_{k}"] = float(v)

        except Exception:

            features.update(_ctd_fallback(sequence))

        return features

    features.update(_aac_fallback(sequence))

    features.update(_paac_fallback(sequence))

    features.update(_ctd_fallback(sequence))

    return features





def build_protein_feature_df(

    uniprot_ids: List[str],

    *,

    force_refetch: bool = False,

    request_delay_s: float = 0.35,

) -> Tuple[pd.DataFrame, pd.DataFrame]:

    """Return (protein_feature_df, failures_df)."""

    failures: List[Dict[str, Any]] = []

    metadata_rows: List[Dict[str, Any]] = []

    records: Dict[str, Dict[str, Any]] = {}



    for uid in tqdm(uniprot_ids, desc="UniProt fetch"):

        rec, err, _ = get_uniprot_record_cached(uid, force_refetch=force_refetch)

        if rec is None:

            failures.append(

                {"uniprot_id": uid, "stage": "fetch", "error": err}

            )

            continue

        records[uid] = rec

        if request_delay_s > 0:

            time.sleep(request_delay_s)



    for uid, rec in records.items():

        try:

            metadata_rows.append(summarize_uniprot_record(uid, rec))

        except Exception as e:

            failures.append({"uniprot_id": uid, "stage": "summarize", "error": str(e)})



    if not metadata_rows:

        return pd.DataFrame(), pd.DataFrame(failures)



    metadata_df = pd.DataFrame.from_records(metadata_rows).set_index("uniprot_id")

    feature_rows: List[Dict[str, Any]] = []

    for uid in metadata_df.index.tolist():

        seq = metadata_df.loc[uid, "sequence"]

        try:

            feature_rows.append(compute_descriptors(uid, seq))

        except Exception as e:

            failures.append({"uniprot_id": uid, "stage": "descriptors", "error": str(e)})



    features_df = pd.DataFrame.from_records(feature_rows).set_index("uniprot_id")

    protein_feature_df = metadata_df.drop(columns=["sequence"]).join(features_df, how="outer")

    failures_df = pd.DataFrame(failures)

    return protein_feature_df, failures_df





# --- Davis mapping ---





def map_target_to_uniprot(target: str, *, request_delay_s: float = 0.35) -> Optional[str]:

    t = str(target).rstrip("p")

    q1 = f"gene:{t} AND organism_id:9606"

    params = {"query": q1, "fields": "accession", "format": "tsv"}

    r = requests.get(UNIPROT_SEARCH_BASE, params=params, timeout=30)

    if request_delay_s > 0:

        time.sleep(request_delay_s)

    if r.status_code == 200 and len(r.text.splitlines()) > 1:

        return r.text.splitlines()[1].strip()

    q2 = f"entry_name:{t.lower()}"

    params["query"] = q2

    r2 = requests.get(UNIPROT_SEARCH_BASE, params=params, timeout=30)

    if request_delay_s > 0:

        time.sleep(request_delay_s)

    if r2.status_code == 200 and len(r2.text.splitlines()) > 1:

        return r2.text.splitlines()[1].strip()

    return None





def load_davis_mapping_cache() -> pd.DataFrame:

    if DAVIS_MAP_CACHE.exists():

        return pd.read_csv(DAVIS_MAP_CACHE, sep="\t")

    return pd.DataFrame(columns=["Target_ID", "uniprot_id", "map_error"])





def save_davis_mapping_cache(df: pd.DataFrame) -> None:

    DAVIS_MAP_CACHE.parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(DAVIS_MAP_CACHE, sep="\t", index=False)





def resolve_davis_uniprot_ids(

    targets: List[str], *, request_delay_s: float = 0.35

) -> pd.DataFrame:

    cache = load_davis_mapping_cache()

    known = set(cache["Target_ID"].astype(str)) if len(cache) else set()

    rows = []

    for tgt in targets:

        tgt_s = str(tgt)

        if tgt_s in known:

            continue

        acc = map_target_to_uniprot(tgt_s, request_delay_s=request_delay_s)

        rows.append(

            {

                "Target_ID": tgt_s,

                "uniprot_id": acc,

                "map_error": None if acc else "mapping_failed",

            }

        )

    if rows:

        cache = pd.concat([cache, pd.DataFrame(rows)], ignore_index=True)

        cache = cache.drop_duplicates(subset=["Target_ID"], keep="last")

        save_davis_mapping_cache(cache)

    return cache[cache["Target_ID"].astype(str).isin([str(t) for t in targets])]





def resolve_uniprot_ids(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:

    out = df.copy()

    if dataset_name == "davis":

        targets = out["Target_ID"].drop_duplicates().tolist()

        mapping = resolve_davis_uniprot_ids(targets, request_delay_s=REQUEST_DELAY_S)

        out = out.merge(mapping, on="Target_ID", how="left")

        if "map_error" not in out.columns:

            out["map_error"] = None

    else:

        out["uniprot_id"] = out["Target_ID"]

        out["map_error"] = np.where(out["Target_ID"].isna(), "null_target_id", None)

    return out





def enrich_dataset(

    dataset_name: str,

    *,

    test_run: bool = False,

    n_test_rows: int = 10,

    force_refetch: bool = False,

    request_delay_s: float = 0.35,

) -> Dict[str, Any]:

    ensure_data_dirs()

    input_path = RAW_DIR / f"{dataset_name}.parquet"

    out_dir = DATA_DIR / ("testrun" if test_run else "processed")

    out_dir.mkdir(parents=True, exist_ok=True)



    raw_df = pd.read_parquet(input_path)

    if test_run:

        work_df = raw_df.head(n_test_rows).copy()

    else:

        work_df = raw_df.copy()



    work_df = resolve_uniprot_ids(work_df, dataset_name)



    feature_cache = CACHE_DIR / f"protein_features_{dataset_name}.parquet"

    failures_all: List[Dict[str, Any]] = []



    ids = (

        work_df["uniprot_id"]

        .dropna()

        .astype(str)

        .str.strip()

        .replace("", np.nan)

        .dropna()

        .unique()

        .tolist()

    )



    use_feature_cache = (

        feature_cache.exists() and not force_refetch and not test_run

    )

    if use_feature_cache:

        protein_feature_df = pd.read_parquet(feature_cache).set_index("uniprot_id")

        fetch_failures = pd.DataFrame()

    else:

        protein_feature_df, fetch_failures = build_protein_feature_df(

            ids, force_refetch=force_refetch, request_delay_s=request_delay_s

        )

        if not test_run and len(protein_feature_df):

            protein_feature_df.reset_index().to_parquet(feature_cache, index=False)



    if len(fetch_failures):

        ff = fetch_failures.copy()

        ff["dataset"] = dataset_name

        failures_all.extend(ff.to_dict("records"))



    if dataset_name == "davis":

        unmapped = work_df[work_df["uniprot_id"].isna()]

        for _, row in unmapped.iterrows():

            failures_all.append(

                {

                    "dataset": dataset_name,

                    "original_target_id": row["Target_ID"],

                    "uniprot_id": None,

                    "stage": "map",

                    "error": row.get("map_error") or "mapping_failed",

                }

            )



    null_targets = work_df[work_df["Target_ID"].isna()]

    for _ in range(len(null_targets)):

        failures_all.append(

            {

                "dataset": dataset_name,

                "original_target_id": None,

                "uniprot_id": None,

                "stage": "map",

                "error": "null_target_id",

            }

        )



    feat_reset = protein_feature_df.reset_index()

    enriched = work_df.merge(feat_reset, on="uniprot_id", how="left", suffixes=("", "_feat"))



    out_path = out_dir / f"{dataset_name}_enriched.parquet"

    enriched.to_parquet(out_path, index=False)



    fail_path = out_dir / f"{dataset_name}_fetch_failures.parquet"

    failures_df = pd.DataFrame(failures_all)

    if len(failures_df):

        failures_df.to_parquet(fail_path, index=False)

    elif fail_path.exists():

        fail_path.unlink()



    n_feat_cols = len([c for c in enriched.columns if c.startswith(("aac_", "paac_", "ctd_"))])

    return {

        "dataset": dataset_name,

        "rows": len(enriched),

        "columns": len(enriched.columns),

        "feature_descriptor_cols": n_feat_cols,

        "unique_uniprot_fetched": len(protein_feature_df),

        "output": str(out_path),

        "failures": len(failures_df),

    }



In [ ]:
# --- Configuration (set before running enrichment) ---
TEST_RUN = True          # True: first 10 rows -> data/testrun/
FULL_RUN = False         # True: full datasets -> data/processed/
N_TEST_ROWS = 10
FORCE_REFETCH = False
REQUEST_DELAY_S = 0.35


## Test run (first 10 rows per dataset)


In [ ]:
ensure_data_dirs()

test_summaries = []
if TEST_RUN:
    for name in DATASETS:
        print(f"\n=== TEST RUN: {name} ===")
        summary = enrich_dataset(
            name,
            test_run=True,
            n_test_rows=N_TEST_ROWS,
            force_refetch=FORCE_REFETCH,
            request_delay_s=REQUEST_DELAY_S,
        )
        test_summaries.append(summary)
        print(summary)
else:
    print("Set TEST_RUN = True to run smoke test.")


## Full-scale run


In [ ]:
full_summaries = []
if FULL_RUN:
    for name in DATASETS:
        print(f"\n=== FULL RUN: {name} ===")
        summary = enrich_dataset(
            name,
            test_run=False,
            force_refetch=FORCE_REFETCH,
            request_delay_s=REQUEST_DELAY_S,
        )
        full_summaries.append(summary)
        print(summary)
else:
    print("Set FULL_RUN = True after test run succeeds.")


## Validation


In [ ]:
out_dir = DATA_DIR / ("testrun" if TEST_RUN and not FULL_RUN else "processed")
if TEST_RUN and not FULL_RUN:
    check_dir = DATA_DIR / "testrun"
elif FULL_RUN:
    check_dir = DATA_DIR / "processed"
else:
    check_dir = DATA_DIR / "testrun"

for name in DATASETS:
    p = check_dir / f"{name}_enriched.parquet"
    if not p.exists():
        print(f"Missing: {p}")
        continue
    df = pd.read_parquet(p)
    n_desc = len([c for c in df.columns if c.startswith(("aac_", "paac_", "ctd_"))])
    print(f"{name}: shape={df.shape}, descriptor_cols={n_desc}")
    display(df.head(2))
    if name == "davis" and "uniprot_id" in df.columns:
        print(df[["Target_ID", "uniprot_id"]].drop_duplicates().head(5))
